# 16 — Análise de Resultados: RAG (todas as variantes)

Avalia as 6 variantes RAG no split de validação (20% do dataset).

**Importante:** O conjunto de avaliação (~4.200 tweets) é diferente dos outros experimentos (~20.813). As métricas não são diretamente comparáveis — o split foi feito para garantir separação limpa entre corpus de retrieval e avaliação.

**Variantes K=3 (top-3 similar global, sem diversidade garantida):**
- `rag_bm25_k3` — BM25 sparse retrieval
- `rag_vector_k3` — Dense retrieval (paraphrase-multilingual-MiniLM-L12-v2)
- `rag_hybrid_qdrant_k3` — Hybrid: dense + TF-IDF sparse, fusão via RRF no Qdrant

**Variantes diversas (top-1 por categoria = 7 exemplos, cobertura garantida de todas as classes):**
- `rag_diverse_bm25_k1` — BM25: top-1 mais similar dentro de cada categoria
- `rag_diverse_vector_k1` — MiniLM: top-1 mais similar dentro de cada categoria
- `rag_diverse_hybrid_k1` — Hybrid Qdrant RRF com filtro por label: top-1 por categoria

In [1]:
from pathlib import Path
import polars as pl
from sklearn.metrics import classification_report, f1_score

ROOT = Path("..")
RESULTS = ROOT / "results" / "full"

VARIANTS = {
    "rag_bm25_k3":            "RAG-BM25 (K=3)",
    "rag_vector_k3":          "RAG-Vector MiniLM (K=3)",
    "rag_hybrid_qdrant_k3":   "RAG-Hybrid Qdrant RRF (K=3)",
    "rag_diverse_bm25_k1":    "RAG-Diverse BM25 (1/classe)",
    "rag_diverse_vector_k1":  "RAG-Diverse Vector MiniLM (1/classe)",
    "rag_diverse_hybrid_k1":  "RAG-Diverse Hybrid Qdrant RRF (1/classe)",
}

In [2]:
summary = []

for key, name in VARIANTS.items():
    path = RESULTS / f"{key}.csv"
    if not path.exists():
        print(f"[SKIP] {path} não encontrado")
        continue

    df = pl.read_csv(path)
    y_true = df["label"].to_list()
    y_pred = df["predicao"].to_list()

    f1_macro   = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    unknown     = (df["predicao"] == "unknown").sum()

    summary.append({"variante": name, "f1_macro": round(f1_macro, 4), "f1_weighted": round(f1_weighted, 4), "unknown": unknown, "tweets": len(df)})

    print(f"\n{'='*60}")
    print(f"  {name}  |  F1-macro: {f1_macro:.4f}  |  F1-weighted: {f1_weighted:.4f}  |  unknown: {unknown}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, zero_division=0))


  RAG-BM25 (K=3)  |  F1-macro: 0.2874  |  F1-weighted: 0.7711  |  unknown: 0
              precision    recall  f1-score   support

  homophobia       0.25      0.29      0.27        34
      insult       0.41      0.29      0.34       301
    misogyny       0.09      0.22      0.12         9
   not_toxic       0.87      0.89      0.88      3387
     obscene       0.36      0.29      0.32       459
      racism       0.04      0.50      0.07         4
  xenophobia       0.00      0.00      0.00         6

    accuracy                           0.78      4200
   macro avg       0.29      0.36      0.29      4200
weighted avg       0.77      0.78      0.77      4200


  RAG-Vector MiniLM (K=3)  |  F1-macro: 0.2647  |  F1-weighted: 0.7689  |  unknown: 1
              precision    recall  f1-score   support

  homophobia       0.38      0.44      0.41        34
      insult       0.37      0.21      0.26       301
    misogyny       0.11      0.33      0.17         9
   not_toxic       0.

In [3]:
print("\n=== Resumo Comparativo (val set 20%) ===")
summary_df = pl.DataFrame(summary).sort("f1_macro", descending=True)
print(summary_df)


=== Resumo Comparativo (val set 20%) ===
shape: (6, 5)
┌─────────────────────────────────┬──────────┬─────────────┬─────────┬────────┐
│ variante                        ┆ f1_macro ┆ f1_weighted ┆ unknown ┆ tweets │
│ ---                             ┆ ---      ┆ ---         ┆ ---     ┆ ---    │
│ str                             ┆ f64      ┆ f64         ┆ i64     ┆ i64    │
╞═════════════════════════════════╪══════════╪═════════════╪═════════╪════════╡
│ RAG-Hybrid Qdrant RRF (K=3)     ┆ 0.2986   ┆ 0.7771      ┆ 0       ┆ 4200   │
│ RAG-Diverse Vector MiniLM (1/c… ┆ 0.289    ┆ 0.7609      ┆ 0       ┆ 4200   │
│ RAG-BM25 (K=3)                  ┆ 0.2874   ┆ 0.7711      ┆ 0       ┆ 4200   │
│ RAG-Diverse Hybrid Qdrant RRF … ┆ 0.2847   ┆ 0.7634      ┆ 0       ┆ 4200   │
│ RAG-Diverse BM25 (1/classe)     ┆ 0.2776   ┆ 0.7591      ┆ 0       ┆ 4200   │
│ RAG-Vector MiniLM (K=3)         ┆ 0.2647   ┆ 0.7689      ┆ 1       ┆ 4200   │
└─────────────────────────────────┴──────────┴─────────────┴────

## Nota sobre comparabilidade

Os resultados do dataset completo (experimentos 04-08) foram calculados em 20.813 tweets sem split.
Aqui avaliamos em ~4.163 tweets (20% estratificado).

Para comparação orientativa com o melhor resultado anterior:
- **FS-v2 2ex+Antibias** no full: F1-macro = **0.3173**

Classes raras no val (esperado):
- `racism`: ~4 exemplos
- `xenophobia`: ~6 exemplos  
- `misogyny`: ~9 exemplos

O F1-macro dessas classes será ruidoso devido ao baixo suporte.